# Bronze Layer - CBBC List Ingestion
Load raw CBBC List CSV files from Azure Data Lake Storage into `bronze.cbbc_list` and record processing results in `bronze.cbbc_list_ingestion_log`.

In [0]:
from datetime import datetime

from pyspark.sql import functions as F
from pyspark.sql.functions import lit, to_date, current_timestamp

## 1. Configure storage access

In [0]:
# Configure OAuth access to Azure Data Lake Storage
service_credential = dbutils.secrets.get(scope="cbbcscope", key="app-secret")

spark.conf.set("fs.azure.account.auth.type.cbbcstakedatalake.dfs.core.windows.net", "OAuth")
spark.conf.set("fs.azure.account.oauth.provider.type.cbbcstakedatalake.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set("fs.azure.account.oauth2.client.id.cbbcstakedatalake.dfs.core.windows.net", "08535731-dcba-457d-a98c-45558d70b8dd")
spark.conf.set("fs.azure.account.oauth2.client.secret.cbbcstakedatalake.dfs.core.windows.net", service_credential)
spark.conf.set("fs.azure.account.oauth2.client.endpoint.cbbcstakedatalake.dfs.core.windows.net", "https://login.windows.net/8b69ada7-612e-4db3-9499-2b3396adf7e0/oauth2/token")

## 2. Identify new source files

In [0]:
# List incoming CSV files from the source folder
raw_cbbc_list = "abfss://source@cbbcstakedatalake.dfs.core.windows.net/cbbc_list/"

files = dbutils.fs.ls(raw_cbbc_list)

incoming_df = spark.createDataFrame(
    [(f.name, f.path) for f in files if f.path.endswith(".csv")],
    ["file_name", "file_path"]
)

# Get files that have not already been processed successfully
processed_df = (
    spark.table("bronze.cbbc_list_ingestion_log")
         .filter(F.col("status") == "SUCCESS")
         .select("file_name")
         .distinct()
)

new_files_df = incoming_df.join(processed_df, on="file_name", how="left_anti")

new_files = [
    {
        "file_name": r["file_name"],
        "file_path": r["file_path"]
    }
    for r in new_files_df.collect()
]

## 3. Process each file and write ingestion log

In [0]:
for file in new_files:
    print(f"Processing {file['file_name']}")

    trade_date_str = None
    status = None
    error_message = None

    try:
        csv_path = file["file_path"]

        df = (
            spark.read
                 .option("header", "true")
                 .format("csv")
                 .load(csv_path)
        )

        df = df.toDF(
            "issuer",
            "code",
            "name",
            "bull_bear",
            "strike",
            "call_level",
            "call_spot_difference_pct",
            "expiry_date",
            "last_trading_day",
            "entitlement_ratio",
            "last_price",
            "issuer_bid",
            "issuer_ask",
            "issuer_bid_ask_spread",
            "change_dollar",
            "change_pct",
            "gearing",
            "premium_pct",
            "outstanding_qty_m",
            "outstanding_qty_pct",
            "outstanding_qty_day_change_m",
            "turnover_kdollar",
            "funding_cost_dollar",
            "funding_cost_pct",
            "underlying_code",
            "underlying_spot",
            "avg_bid_ask_spread_tick",
            "one_tick_spread_duration_pct",
            "liquidity",
            "listing_date",
            "quote"
        )

        # Extract trade date from the file name
        date_YYYYMMDD = file["file_name"].split("_")[2].split(".")[0]
        trade_date_str = datetime.strptime(date_YYYYMMDD, "%Y%m%d").strftime("%Y-%m-%d")

        # Add ingestion metadata
        df = df.withColumn("trade_date", to_date(lit(trade_date_str)))
        df = df.withColumn("processed_at", current_timestamp())

        (
            df.write
              .format("delta")
              .mode("overwrite")
              .option("replaceWhere", f"trade_date = '{trade_date_str}'")
              .saveAsTable("bronze.cbbc_list")
        )

        status = "SUCCESS"
        error_message = "-"
        print("SUCCESS")

    except Exception as e:
        print("FAILED")
        print(e)

        if not trade_date_str:
            trade_date_str = "-"

        status = "FAILED"
        error_message = str(e)[:5000]

    # Write processing result to ingestion log
    log_df = spark.createDataFrame(
        [(file["file_name"], trade_date_str, status, error_message)],
        ["file_name", "trade_date", "status", "error_message"]
    ).withColumn("processed_at", current_timestamp())

    log_df.write.mode("append").saveAsTable("bronze.cbbc_list_ingestion_log")